# T015: D3 baseline training (Colab T4)

Trains `configs/train/baseline.yaml` into `artifacts/runs/<RUN_ID>/` and copies the run
folder to Google Drive every 10 minutes, so a dropped session still leaves evidence.
Steps match `docs/colab-setup.md` ("D3 baseline training on Colab").

1. **Runtime > Change runtime type > T4 GPU.**
2. Run the cells in order. Keep this tab open while training runs.
3. If the session drops, do **not** resume: set `RUN_ID` to the next unused ID
   (`baseline-02`, ...) and start again. The partial Drive folder is the evidence of
   the failed run.

In [ ]:
RUN_ID = 'baseline-01'  # change to baseline-02, ... only for a retry
BRANCH = '001-driving-object-detection-us3'
REPO = '/content/cs4337_project'
DRIVE_RUNS = '/content/drive/MyDrive/cs4337/runs'
!nvidia-smi
!python --version

## 1. Clone the branch and mount Drive

In [ ]:
import os, subprocess
from pathlib import Path
if not Path(REPO).exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH,
                    'https://github.com/DiSaber/cs4337_project.git', REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_RUNS, exist_ok=True)
assert not Path(DRIVE_RUNS, RUN_ID).exists(), f'{RUN_ID} already exists on Drive; pick a new RUN_ID'

## 2. Install, download data, prepare (about 15 min, 5.9 GB)

The CUDA lock is for Python 3.13. If the first cell printed another version, stop and record it.

In [ ]:
%%bash
set -euo pipefail
cd /content/cs4337_project
python -m venv --without-pip .venv-colab
python -m pip --python .venv-colab/bin/python install -q -r requirements/cuda-lock.txt
python -m pip --python .venv-colab/bin/python install -q --no-deps -e .
python -m pip --python .venv-colab/bin/python check
.venv-colab/bin/python -m driving_detection assets sync --config configs/assets.yaml
.venv-colab/bin/python -m driving_detection check --config configs/assets.yaml --device cuda:0
.venv-colab/bin/python -m driving_detection data prepare --config configs/data/bdd-subset.yaml

## 3. Check that preparation reproduced the frozen D2 profile

Stops with an error if any hash differs. Do not train on different data.

In [ ]:
import json
EXPECTED = {
    'manifest.train.jsonl': '0f3b35bfc1ded48c55ebf4d6b30cb8e55089f4e51461fd9865d36e2ca2a15251',
    'manifest.val.jsonl': '73d93c05c045119cf276fd7f131fbd3181503bbb4103e335119a26cc72f495e0',
    'manifest.test.jsonl': '42286df26bc61943244c51aa51e5702a65dec4e8a7a3d2611a08abe14d6a76fb',
    'coco.val.json': '26f8e1b948a83346336d20a64ad3c82c1c32af9f4c75b49f6b79ebf6758d38f8',
    'yolo/dataset.yaml': 'fd880cae4bd4604e5c458d0c9c7c8e7ebb9230251e9b2ae6917c0a6b4f4e4b8d',
    'yolo/labels/train': 'b351ebaca4af2dd98dc716a25a61292717a54a5fd297663139ba1081d547fed8',
    'yolo/labels/val': '53b2b486636e0aaabbf05e28812fe6d7b7b1e1582f0451d3490375636075263d',
}
exports = json.load(open(f'{REPO}/data/prepared/bdd-subset-v2/profile.json'))['exports']
bad = []
for name, want in EXPECTED.items():
    got = exports[name]
    got = got['tree_sha256'] if isinstance(got, dict) else got
    print('OK ' if got == want else 'BAD', name, got[:12])
    if got != want:
        bad.append(name)
assert not bad, f'Profile differs from D2 for {bad}; stop and record this'
print('All hashes match the frozen D2 profile.')

## 4. Start the Drive copy loop (every 10 minutes)

In [ ]:
import subprocess
src = f'{REPO}/artifacts/runs/{RUN_ID}'
loop = (f'while true; do [ -d {src} ] && rsync -a {src} {DRIVE_RUNS}/; sleep 600; done')
sync_proc = subprocess.Popen(['bash', '-c', loop],
                             stdout=open('/content/drive-sync.log', 'w'), stderr=subprocess.STDOUT)
print('copy loop pid', sync_proc.pid)

## 5. Train (up to about 8 GPU-hours)

Keep the tab open. Output streams below.

In [ ]:
!cd {REPO} && .venv-colab/bin/python -m driving_detection train \
  --profile data/prepared/bdd-subset-v2/profile.json \
  --config configs/train/baseline.yaml --run-id {RUN_ID} --device cuda:0; echo "train exit code: $?"

## 6. Final copy to Drive and quick check

Run this whether training succeeded or failed.

In [ ]:
import json, subprocess
sync_proc.terminate()
subprocess.run(['rsync', '-a', f'{REPO}/artifacts/runs/{RUN_ID}', f'{DRIVE_RUNS}/'], check=True)
!ls -la {DRIVE_RUNS}/{RUN_ID} {DRIVE_RUNS}/{RUN_ID}/weights
run = json.load(open(f'{DRIVE_RUNS}/{RUN_ID}/run.json'))
print('status:', run.get('status'), '| label:', run.get('label'), '| head:', run.get('head'))
!wc -l {DRIVE_RUNS}/{RUN_ID}/results.csv

Expected: `status: completed`, the baseline label, 21 lines in `results.csv`
(header + 20 epochs), and a head with car, pedestrian, traffic_light.

Optional: also download a zip directly (about 30 MB).

In [ ]:
import shutil
from google.colab import files
bundle = shutil.make_archive(f'/content/{RUN_ID}', 'zip',
                             root_dir=f'{REPO}/artifacts/runs', base_dir=RUN_ID)
files.download(bundle)